<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        2.3.2 Técnicas de Visualización de Datos Numéricos 📊
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Visual Analytics and Critical Thinking
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 05
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Visual%20Analytics%20and%20Critical%20Thinking/05%20-%20Visualizacion%20de%20Datos%20Numericos/02_Tecnicas_Visualizacion_Numericos.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

1. Visualizar **tendencias a lo largo del tiempo** con gráficos de línea (line graphs).
2. Visualizar **distribuciones** con histogramas, KDE (Kernel Density Estimation) y ECDF.
3. Detectar **outliers y dispersión** con box plots y violin plots.
4. Explorar **relaciones bivariadas** con scatter plots y analizar **correlaciones multivariadas** con heatmaps.
5. Mapear cada visualización a su equivalente en **Power BI** y **Tableau**.

## Recursos Recomendados 📚

- Few, S. (2009). *Now You See It: Simple Visualization Techniques for Quantitative Analysis*.
- Documentación de [Seaborn: visualizing distributions](https://seaborn.pydata.org/tutorial/distributions.html).
- Documentación de [Plotly Express](https://plotly.com/python/plotly-express/).

In [ ]:
# ============================================================
# Setup: Dataset de trabajo
# Dataset: California Housing (incluido en scikit-learn)
# Variables numéricas reales de casas en California
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

# Cargar dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame
df.rename(columns={
    'MedHouseVal': 'precio_mediano',
    'MedInc': 'ingreso_mediano',
    'HouseAge': 'edad_casa',
    'AveRooms': 'habitaciones_prom',
    'AveBedrms': 'dormitorios_prom',
    'Population': 'poblacion',
    'AveOccup': 'ocupacion_prom',
    'Latitude': 'latitud',
    'Longitude': 'longitud'
}, inplace=True)
df['precio_mediano'] = df['precio_mediano'] * 100000  # En dólares reales

print(f"Dataset California Housing: {df.shape[0]:,} registros x {df.shape[1]} variables")
print("\nEstadísticas descriptivas:")
print(df[['precio_mediano', 'ingreso_mediano', 'edad_casa', 'habitaciones_prom']].describe().round(2))

---
## 1. Line Graphs: visualizando tendencias 📈

El **gráfico de línea** es la técnica preferida cuando el eje X representa una secuencia **ordenada**, típicamente tiempo. Conecta observaciones sucesivas para resaltar la **tendencia**, no solo el valor puntual.

> ⚠️ Regla de oro: solo conecta con una línea puntos que tengan un orden natural (tiempo, secuencia). Conectar categorías sin orden (como "regiones") con una línea es uno de los errores más comunes en visualización.

In [ ]:
# ============================================================
# Line graph: tendencia simulada de precio promedio de vivienda por año
# (el dataset original no trae fecha, así que simulamos una serie temporal
#  razonable para ilustrar la técnica)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(21)
anios = np.arange(2014, 2024)
precio_promedio_anual = 200 + np.cumsum(np.random.normal(8, 6, len(anios)))  # miles $

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(anios, precio_promedio_anual, marker='o', color='#1e3a8a', linewidth=2.5)
ax.fill_between(anios, precio_promedio_anual, alpha=0.08, color='#1e3a8a')
ax.set_title('Line Graph: Tendencia del precio promedio de vivienda (simulado)', fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('Precio promedio (miles $)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("📊 En Power BI: Visual nativo 'Gráfico de líneas'.")
print("📈 En Tableau: Arrastra una fecha/orden a Columns y una medida a Rows → Show Me → Line.")

---
## 2. Visualizando Distribuciones: Histograma, KDE y ECDF 📊

La **distribución** es la característica más fundamental de una variable numérica. Responde: ¿dónde se concentran los valores? ¿Qué forma tiene la distribución?

| Gráfico | Qué muestra | Mejor para |
|---|---|---|
| **Histograma** | Frecuencia de valores en bins | Vista rápida de la forma |
| **KDE** | Curva de densidad suavizada | Comparar múltiples distribuciones |
| **ECDF** | Proporción acumulada de datos | Percentiles y comparación robusta |

In [ ]:
# ============================================================
# Histograma + KDE + ECDF: la trinidad de distribuciones
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribución del Precio Mediano de Casas — California Housing',
             fontsize=13, fontweight='bold')

precios = df['precio_mediano'] / 1000  # En miles

# 1. Histograma
axes[0].hist(precios, bins=50, color='#3b82f6', alpha=0.8, edgecolor='white')
axes[0].axvline(precios.mean(), color='#dc2626', linestyle='--', linewidth=2,
                label=f'Media: ${precios.mean():.0f}k')
axes[0].axvline(precios.median(), color='#16a34a', linestyle='--', linewidth=2,
                label=f'Mediana: ${precios.median():.0f}k')
axes[0].set_xlabel('Precio Mediano (miles $)', fontsize=10)
axes[0].set_ylabel('Frecuencia', fontsize=10)
axes[0].set_title('① Histograma', fontweight='bold')
axes[0].legend(fontsize=9)

# 2. KDE (Kernel Density Estimation)
precios.plot.kde(ax=axes[1], color='#3b82f6', linewidth=2.5)
axes[1].fill_between(axes[1].lines[0].get_xdata(), axes[1].lines[0].get_ydata(),
                     alpha=0.1, color='#3b82f6')
axes[1].axvline(precios.mean(), color='#dc2626', linestyle='--', linewidth=2)
axes[1].axvline(precios.median(), color='#16a34a', linestyle='--', linewidth=2)
axes[1].set_xlabel('Precio Mediano (miles $)', fontsize=10)
axes[1].set_ylabel('Densidad', fontsize=10)
axes[1].set_title('② KDE — Curva de Densidad', fontweight='bold')
axes[1].set_xlim(-50, 600)

# 3. ECDF
sorted_prices = np.sort(precios)
ecdf = np.arange(1, len(sorted_prices) + 1) / len(sorted_prices)
axes[2].plot(sorted_prices, ecdf, color='#3b82f6', linewidth=2)
axes[2].axhline(0.5, color='#16a34a', linestyle='--', linewidth=1.5,
                label=f'P50 (Mediana): ${precios.quantile(0.5):.0f}k')
axes[2].axhline(0.75, color='#f59e0b', linestyle='--', linewidth=1.5,
                label=f'P75: ${precios.quantile(0.75):.0f}k')
axes[2].axhline(0.25, color='#ef4444', linestyle='--', linewidth=1.5,
                label=f'P25: ${precios.quantile(0.25):.0f}k')
axes[2].set_xlabel('Precio Mediano (miles $)', fontsize=10)
axes[2].set_ylabel('Proporción acumulada', fontsize=10)
axes[2].set_title('③ ECDF — Distribución Acumulada', fontweight='bold')
axes[2].legend(fontsize=8)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("\n📊 En Power BI: Histograma → Visual 'Histograma de Python' o usar bins manuales.")
print("   KDE: No nativo — usar visual de Python embebido o AppSource.")
print("\n📈 En Tableau: Show Me → Histogram. Ajustar tamaño del bin en el eje X.")

---
##### 🛠️ Práctica 1: Histograma y KDE: ancho de bin y ancho de banda

Esta práctica es autocontenida (no usa `df`). Genera una muestra **bimodal** (dos grupos mezclados) y mira cómo el ancho de bin del histograma y el ancho de banda del KDE pueden ocultar o mostrar los dos picos:

```python
rng = np.random.default_rng(7)
x = np.concatenate([rng.normal(-2, 1, 1000), rng.normal(2.5, 1, 1000)])
```

1. Crea `fig, axes = plt.subplots(2, 3, figsize=(15, 7))`.
2. Fila superior: histogramas de `x` con `density=True` y `bins=3`, `30` y `300`. Guarda los conteos de cada uno en `n_bins`.
3. Fila inferior: KDE con `from scipy.stats import gaussian_kde` evaluado en `grilla = np.linspace(-8, 9, 400)` para los anchos de banda `bw_method = 0.05`, `0.3` y `2.0`.
4. Define `n_picos(y)` con `np.sum((y[1:-1] > y[:-2]) & (y[1:-1] > y[2:]))` y cuenta los máximos locales de cada KDE.
5. Verifica con `assert`: `n_bins` tiene 3, 30 y 300 elementos; el KDE con `0.05` tiene **más de 2** picos (ruido), con `0.3` tiene **exactamente 2** y con `2.0` tiene **1** (sobresuavizado).

In [ ]:
# Escribe tu código aquí
from scipy.stats import gaussian_kde

rng = np.random.default_rng(7)
x = np.concatenate([rng.normal(-2, 1, 1000), rng.normal(2.5, 1, 1000)])
grilla = np.linspace(-8, 9, 400)

# fig, axes = plt.subplots(2, 3, figsize=(15, 7))
# for ax, b in zip(axes[0], [3, 30, 300]):
#     conteos, _, _ = ax.hist(x, bins=b, density=True)
#     ...

def n_picos(y):
    return int(np.sum((y[1:-1] > y[:-2]) & (y[1:-1] > y[2:])))

# for ax, bw in zip(axes[1], [0.05, 0.3, 2.0]):
#     y = gaussian_kde(x, bw_method=bw)(grilla)
#     ax.plot(grilla, y)
#     picos[bw] = n_picos(y)

# assert picos[0.05] > 2 and picos[0.3] == 2 and picos[2.0] == 1

<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
from scipy.stats import gaussian_kde

rng = np.random.default_rng(7)
x = np.concatenate([rng.normal(-2, 1, 1000), rng.normal(2.5, 1, 1000)])
grilla = np.linspace(-8, 9, 400)

def n_picos(y):
    return int(np.sum((y[1:-1] > y[:-2]) & (y[1:-1] > y[2:])))

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
n_bins = {}
picos = {}

# Fila superior: mismo dato, tres anchos de bin
for ax, b in zip(axes[0], [3, 30, 300]):
    conteos, _, _ = ax.hist(x, bins=b, density=True, color='#3b82f6', edgecolor='white')
    n_bins[b] = len(conteos)
    ax.set_title(f'Histograma con {b} bins', fontweight='bold')

# Fila inferior: mismo dato, tres anchos de banda
for ax, bw in zip(axes[1], [0.05, 0.3, 2.0]):
    y = gaussian_kde(x, bw_method=bw)(grilla)
    picos[bw] = n_picos(y)
    ax.plot(grilla, y, color='#1e3a8a', linewidth=2)
    ax.fill_between(grilla, y, alpha=0.1, color='#1e3a8a')
    ax.set_title(f'KDE bw = {bw} ({picos[bw]} picos)', fontweight='bold')

for ax in axes.ravel():
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

assert list(n_bins) == [3, 30, 300] and [n_bins[b] for b in (3, 30, 300)] == [3, 30, 300]
assert picos[0.05] > 2
assert picos[0.3] == 2
assert picos[2.0] == 1
print("Poco suavizado = ruido; demasiado suavizado = se pierde un grupo entero. Hay que probar varios anchos.")
```
</details>

---

---
## 3. Box Plot y Violin Plot: Detección de Outliers 🎻

In [ ]:
# ============================================================
# Box Plot y Violin Plot comparativos
# Seaborn + Plotly interactivo
# ============================================================

# Crear grupos de ingreso para comparar
df['grupo_ingreso'] = pd.cut(df['ingreso_mediano'],
                              bins=[0, 2, 4, 6, 20],
                              labels=['Bajo (<2)', 'Medio (2-4)', 'Alto (4-6)', 'Muy Alto (>6)'])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Precio de Casas por Grupo de Ingreso — Box vs Violin',
             fontsize=13, fontweight='bold')

# Box Plot
sns.boxplot(data=df, x='grupo_ingreso', y='precio_mediano',
            ax=axes[0], palette='Set2', linewidth=1.5)
axes[0].set_title('Box Plot: Mediana, IQR y Outliers', fontweight='bold')
axes[0].set_xlabel('Grupo de Ingreso Mediano', fontsize=10)
axes[0].set_ylabel('Precio Mediano ($)', fontsize=10)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Violin Plot
sns.violinplot(data=df, x='grupo_ingreso', y='precio_mediano',
               ax=axes[1], palette='Set2', inner='quartile', linewidth=1.5)
axes[1].set_title('Violin Plot: Distribución completa + cuartiles', fontweight='bold')
axes[1].set_xlabel('Grupo de Ingreso Mediano', fontsize=10)
axes[1].set_ylabel('Precio Mediano ($)', fontsize=10)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("📊 En Power BI: No hay box plot nativo — usar 'Box and Whisker Chart' de AppSource.")
print("   Alternativa: usar visual de Python (requiere R/Python habilitado en el servicio)")
print("\n📈 En Tableau: Show Me → Box-and-Whisker Plot. Requiere al menos una dimensión y medida.")

---
##### 🛠️ Práctica 2: Box plot y regla del IQR

Esta práctica es autocontenida. Compara el monto de compra de tres sucursales y comprueba que los puntos que dibuja el box plot son los *outliers* de la regla del IQR:

```python
rng = np.random.default_rng(11)
sucursales = ['Norte', 'Centro', 'Sur']
datos = pd.DataFrame({
    'sucursal': np.repeat(sucursales, 200),
    'monto': np.concatenate([rng.lognormal(4.0, 0.4, 200),
                             rng.lognormal(4.3, 0.4, 200),
                             rng.lognormal(4.0, 0.8, 200)]),
})
```

1. Arma `grupos = [datos.loc[datos['sucursal'] == s, 'monto'] for s in sucursales]`.
2. Crea `fig, ax = plt.subplots(figsize=(7, 4.5))`, dibuja `bp = ax.boxplot(grupos)` y pon los nombres con `ax.set_xticklabels(sucursales)`.
3. Para cada grupo calcula con pandas `q1`, `q3`, `iqr = q3 - q1`; guarda los IQR en la lista `iqrs` y cuenta los valores fuera de `[q1 - 1.5*iqr, q3 + 1.5*iqr]` en la lista `n_out_regla`.
4. Cuenta los puntos que dibujó Matplotlib: `n_out_grafico = [len(f.get_ydata()) for f in bp['fliers']]`.
5. Verifica con `assert`: ambas listas son iguales, la mediana dibujada de cada caja coincide con `g.median()` y la sucursal con mayor IQR (la más dispersa) es `'Sur'`.

In [ ]:
# Escribe tu código aquí

rng = np.random.default_rng(11)
sucursales = ['Norte', 'Centro', 'Sur']
datos = pd.DataFrame({
    'sucursal': np.repeat(sucursales, 200),
    'monto': np.concatenate([rng.lognormal(4.0, 0.4, 200),
                             rng.lognormal(4.3, 0.4, 200),
                             rng.lognormal(4.0, 0.8, 200)]),
})

# grupos = [datos.loc[datos['sucursal'] == s, 'monto'] for s in sucursales]

# fig, ax = plt.subplots(figsize=(7, 4.5))
# bp = ax.boxplot(grupos)
# ax.set_xticklabels(sucursales)

# iqrs, n_out_regla = [], []
# for g in grupos:
#     q1, q3 = g.quantile([0.25, 0.75])
#     iqr = q3 - q1
#     iqrs.append(iqr)
#     n_out_regla.append(int(((g < q1 - 1.5 * iqr) | (g > q3 + 1.5 * iqr)).sum()))

# n_out_grafico = [len(f.get_ydata()) for f in bp['fliers']]

# assert n_out_regla == n_out_grafico

<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
rng = np.random.default_rng(11)
sucursales = ['Norte', 'Centro', 'Sur']
datos = pd.DataFrame({
    'sucursal': np.repeat(sucursales, 200),
    'monto': np.concatenate([rng.lognormal(4.0, 0.4, 200),
                             rng.lognormal(4.3, 0.4, 200),
                             rng.lognormal(4.0, 0.8, 200)]),
})

# 1) Un grupo por sucursal
grupos = [datos.loc[datos['sucursal'] == s, 'monto'] for s in sucursales]

# 2) Box plot
fig, ax = plt.subplots(figsize=(7, 4.5))
bp = ax.boxplot(grupos)
ax.set_xticklabels(sucursales)
ax.set_xlabel('Sucursal')
ax.set_ylabel('Monto de compra')
ax.set_title('Monto de compra por sucursal', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# 3) Outliers según la regla del IQR (1.5 * IQR)
iqrs, n_out_regla = [], []
for g in grupos:
    q1, q3 = g.quantile([0.25, 0.75])
    iqr = q3 - q1
    iqrs.append(iqr)
    n_out_regla.append(int(((g < q1 - 1.5 * iqr) | (g > q3 + 1.5 * iqr)).sum()))

# 4) Puntos que dibujó Matplotlib como outliers
n_out_grafico = [len(f.get_ydata()) for f in bp['fliers']]
print('Regla IQR:', dict(zip(sucursales, n_out_regla)))

# 5) Verificación
assert n_out_regla == n_out_grafico
for g, m in zip(grupos, bp['medians']):
    assert np.isclose(m.get_ydata()[0], g.median())
assert sucursales[int(np.argmax(iqrs))] == 'Sur'
print("Los puntos del box plot son exactamente los valores fuera de 1.5 * IQR; la caja más alta (mayor IQR) es la de la sucursal más dispersa.")
```
</details>

---

---
## 4. Scatter Plot y Correlaciones: Relaciones entre Variables 🔗

In [ ]:
# ============================================================
# Heatmap de Correlación + Scatter con línea de tendencia
# Análisis multivariado de variables numéricas
# ============================================================

vars_num = ['precio_mediano', 'ingreso_mediano', 'edad_casa',
            'habitaciones_prom', 'poblacion']

# Matriz de correlación
corr_matrix = df[vars_num].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Análisis de Correlación — Dataset California Housing',
             fontsize=13, fontweight='bold')

# Heatmap de correlación
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu',
            center=0, vmin=-1, vmax=1, ax=axes[0],
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[0].set_title('Heatmap de Correlación (Pearson)', fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, fontsize=8)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0, fontsize=8)

# Scatter con regresión: precio vs ingreso
sample = df.sample(1000, random_state=42)
scatter = axes[1].scatter(sample['ingreso_mediano'], sample['precio_mediano'] / 1000,
                           c=sample['edad_casa'], cmap='viridis',
                           alpha=0.6, s=30)
plt.colorbar(scatter, ax=axes[1], label='Edad de la Casa (años)')

x_fit = sample['ingreso_mediano'].values
y_fit = (sample['precio_mediano'] / 1000).values
coefs = np.polyfit(x_fit, y_fit, 1)
x_line = np.linspace(x_fit.min(), x_fit.max(), 100)
axes[1].plot(x_line, np.polyval(coefs, x_line), 'r--', linewidth=2, label='Tendencia lineal')

r = np.corrcoef(x_fit, y_fit)[0, 1]
axes[1].set_xlabel('Ingreso Mediano', fontsize=10)
axes[1].set_ylabel('Precio Mediano (miles $)', fontsize=10)
axes[1].set_title(f'Scatter: Ingreso vs Precio | r = {r:.3f}', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("\n📊 En Power BI: Heatmap → AppSource 'Table Heatmap'. Scatter nativo disponible.")
print("   Agregar línea de tendencia en scatter: Formatear visual → 'Serie' → 'Línea de tendencia'")
print("\n📈 En Tableau: Analytics Panel → arrastrar 'Trend Line' al scatter plot.")

---
##### 🛠️ Práctica 3: Scatter con regresión y matriz de correlación

Esta práctica es autocontenida (usa `np`, `pd`, `plt` y `sns` del setup). Con datos de estudiantes simulados:

```python
rng = np.random.default_rng(3)
n = 200
horas = rng.uniform(0, 10, n)
df_est = pd.DataFrame({
    'horas_estudio': horas,
    'nota': 2 + 0.4 * horas + rng.normal(0, 0.5, n),
    'horas_sueno': rng.normal(7, 1, n),
    'cafes': rng.poisson(2, n),
})
```

1. Calcula `corr = df_est.corr()` y dibújala con `sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu', center=0, vmin=-1, vmax=1, ax=ax1)` en `fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))`.
2. En `ax2` haz el scatter de `horas_estudio` (X) contra `nota` (Y) y agrega la recta ajustada con `pendiente, intercepto = np.polyfit(...)`.
3. Calcula `r = np.corrcoef(...)[0, 1]` y ponlo en el título del scatter.
4. Verifica con `assert`: `corr` es 4x4 con diagonal igual a 1; `r` coincide con `corr.loc['horas_estudio', 'nota']`; la pendiente está entre 0.3 y 0.5; y, sin contar la propia `nota`, la variable más correlacionada con `nota` (en valor absoluto) es `'horas_estudio'`.

In [ ]:
# Escribe tu código aquí

rng = np.random.default_rng(3)
n = 200
horas = rng.uniform(0, 10, n)
df_est = pd.DataFrame({
    'horas_estudio': horas,
    'nota': 2 + 0.4 * horas + rng.normal(0, 0.5, n),
    'horas_sueno': rng.normal(7, 1, n),
    'cafes': rng.poisson(2, n),
})

# corr = df_est.corr()
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
# sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu', center=0, vmin=-1, vmax=1, ax=ax1)

# ax2.scatter(df_est['horas_estudio'], df_est['nota'], alpha=0.6)
# pendiente, intercepto = np.polyfit(df_est['horas_estudio'], df_est['nota'], 1)
# ...

# r = np.corrcoef(df_est['horas_estudio'], df_est['nota'])[0, 1]

# assert corr.shape == (4, 4)

<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
rng = np.random.default_rng(3)
n = 200
horas = rng.uniform(0, 10, n)
df_est = pd.DataFrame({
    'horas_estudio': horas,
    'nota': 2 + 0.4 * horas + rng.normal(0, 0.5, n),
    'horas_sueno': rng.normal(7, 1, n),
    'cafes': rng.poisson(2, n),
})

# 1) Matriz de correlación
corr = df_est.corr()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu', center=0, vmin=-1, vmax=1,
            square=True, ax=ax1)
ax1.set_title('Matriz de correlación (Pearson)', fontweight='bold')

# 2) Scatter + recta de regresión
x_e, y_n = df_est['horas_estudio'], df_est['nota']
ax2.scatter(x_e, y_n, color='#7c3aed', alpha=0.6, s=30)
pendiente, intercepto = np.polyfit(x_e, y_n, 1)
x_line = np.linspace(x_e.min(), x_e.max(), 100)
ax2.plot(x_line, intercepto + pendiente * x_line, 'r--', linewidth=2, label='Tendencia lineal')

# 3) Coeficiente r en el título
r = np.corrcoef(x_e, y_n)[0, 1]
ax2.set_title(f'Horas de estudio vs nota | r = {r:.3f}', fontweight='bold')
ax2.set_xlabel('Horas de estudio')
ax2.set_ylabel('Nota')
ax2.legend()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# 4) Verificación
assert corr.shape == (4, 4) and np.allclose(np.diag(corr), 1)
assert np.isclose(r, corr.loc['horas_estudio', 'nota'])
assert 0.3 < pendiente < 0.5
assert corr['nota'].drop('nota').abs().idxmax() == 'horas_estudio'
print(f"Cada hora extra de estudio se asocia con +{pendiente:.2f} puntos de nota (r = {r:.2f}).")
```
</details>

---

---
## 5. Tabla Resumen: Técnicas de Visualización Numérica 🎯

| Objetivo | Gráfico | Python | Power BI | Tableau |
|---|---|---|---|---|
| Tendencia en el tiempo | Line graph | `plt.plot()` | Gráfico de líneas | Line |
| Distribución univariada | Histograma | `plt.hist()`, `sns.histplot()` | Histograma (AppSource) | Histogram |
| Distribución suavizada | KDE | `sns.kdeplot()` | Python visual | AppSource |
| Percentiles y outliers | Box Plot | `sns.boxplot()`, `px.box()` | AppSource | Box-and-Whisker |
| Distribución + densidad | Violin | `sns.violinplot()` | Python visual | No nativo |
| Relación 2 variables | Scatter | `px.scatter()` | Gráfico de dispersión | Scatter Plot |
| Correlaciones múltiples | Heatmap | `sns.heatmap()` | AppSource: Table Heatmap | Heatmap |

> 🚀 **Siguiente cuaderno:** [Estudios de Caso de Visualización de Datos Numéricos](03_Estudios_de_Caso_Numericos.ipynb) — aplicamos estas técnicas a un caso completo con ventas mensuales y tiempos de respuesta de un sistema.

## Resumen y Puntos Clave 🎯

- Los **line graphs** son la técnica correcta para mostrar tendencias en secuencias ordenadas (tiempo).
- Histograma, KDE y ECDF son tres formas complementarias de ver la **distribución** de una variable.
- Box plot y violin plot permiten comparar distribuciones entre grupos y **detectar outliers**.
- Scatter plots revelan **relaciones bivariadas**; los heatmaps de correlación resumen relaciones entre muchas variables a la vez.
- Cada técnica tiene un equivalente directo en Power BI y Tableau, aunque algunas (KDE, violin) requieren visuales adicionales de AppSource.

### 🧠 Autoevaluación

1. ¿Por qué no deberías usar un gráfico de línea para conectar los precios promedio de distintas regiones (sin orden temporal)?
2. ¿Qué información adicional te da un violin plot que un box plot no muestra directamente?
3. En el heatmap de correlación, ¿qué significa un valor cercano a -1 entre dos variables?
4. Si tuvieras que elegir entre histograma y KDE para comparar la distribución de precios de 5 ciudades en el mismo gráfico, ¿cuál elegirías y por qué?